**Kepler's Exoplanet Data Cleaning**

Objective: "To clean and standardize a dataset of 10,000 messy records captured by kepler for research."

Key skills demonstrated: Python, Pandas, Git/Github

Setting up the requirements:

In [5]:
# BLOCK_01
# Drive access
from google.colab import drive, files, userdata
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
# BLOCK_02
# import required libraries
import os
import shutil

# Define your designated workspace folder
target_folder = input('Enter folder name (e.g., 01_NASA_Exoplanet_Archive_(Astrophysics)):')

# Automatically create the folder if it doesn't exist
os.makedirs(target_folder, exist_ok=True)

# 1. Upload the file
print("Please upload your CSV file:")
uploaded = files.upload()

uploaded_filename = list(uploaded.keys())[0]
output_filename = f"cleaned_{uploaded_filename}"

# 2. Set the paths for your input and output files
input_path = os.path.join(target_folder, uploaded_filename)
output_path = os.path.join(target_folder, output_filename)

# 3. Move the uploaded file into the target folder
for filename in uploaded.keys():
    source_path = filename
    destination_path = os.path.join(target_folder, filename)
    shutil.move(source_path, destination_path)
    print(f"User uploaded file {filename} with length {len(uploaded[filename])} bytes")
    print(f"Success: Moved '{filename}' to '/{target_folder}'")

Enter folder name (e.g., 01_NASA_Exoplanet_Archive_(Astrophysics)):01_NASA_Exoplanet_Archive_(Astrophysics)
Please upload your CSV file:


IndexError: list index out of range

**Raw data (current state)**

the problems:

- 100% Missing Data: The columns ```koi_teq_err1``` and ```koi_teq_err2``` are completely empty (```9,564``` missing values each).

- Columns like ```rowid```, ```kepid```, and ```kepoi_name``` are merely internal IDs or row numbers.

Handling Missing Values (```NaNs```):

- ```kepler_name```: This column is missing ```7,270``` values. Should either drop this column entirely (since ```kepoi_name``` serves as a unique ID anyway) or fill the missing values with a string like ```"Unknown"```.

- ```koi_score```: Missing ```1,510 values```, imputing them (e.g., using the ```median```)

- Several error columns (e.g., ```koi_steff_err1/2```, ```koi_srad_err1/2```, ```koi_slogg_err1/2```) are missing around ```450``` to ```480``` values. Imputing these missing values with the median of their respective columns.

Extreme Outliers:

- ```koi_period``` (Orbital Period): The ```median``` is ```~9.7 days```, and ```99%``` of the data is under ```550 days```. However, the maximum value is ```129,995 days```.

- ```koi_prad``` (Planet Radius): The ```median``` is ```2.39 Earth radii```, but the ```maximum``` is a massive ```200,346 Earth radii```.

- ```koi_insol``` (Insolation Flux): The ```maximum``` is ```nearly 10.9 million times Earth's flux```, which is exceptionally high.

Categorical Encoding:

- ```koi_disposition & koi_pdisposition```: These are likely target variables (containing values like ```CONFIRMED```, ```CANDIDATE```, ```FALSE POSITIVE```). Will need to encode these into numerical values using Label Encoding or One-Hot Encoding if it is planned to feed this data into a machine learning algorithm.

- ```koi_tce_delivname```: This categorical column has 3 unique values representing data delivery pipelines.

**Before :**

In [ ]:
# BLOCK_03
# Remember, setting up the required requirements is imperative
import pandas as pd
import numpy as np

# reading the uploaded file
df = pd.read_csv(input_path)

df.sample()

In [ ]:
# BLOCK_04
import pandas as pd
import numpy as np

# 1. Ingestion (Handling spaces automatically)
df = pd.read_csv(input_path, skipinitialspace=True)

# -----------------------------------------------------------------------------

# 2. Dropping Useless and Empty Columns
cols_to_drop = ['rowid', 'kepid', 'kepoi_name', 'koi_teq_err1', 'koi_teq_err2']
df = df.drop(columns=cols_to_drop)

# 3. Handling Missing Values
df['kepler_name'] = df['kepler_name'].fillna("Unknown")

error_cols = [col for col in df.columns if 'err1' in col or 'err2' in col]
df[error_cols] = df[error_cols].fillna(df[error_cols].median())

df = df.dropna(subset=['koi_disposition'])
df['koi_score'] = df['koi_score'].fillna(df['koi_score'].median())

# 4. Outlier Capping (Winsorization)
outlier_cols = ['koi_period', 'koi_prad', 'koi_insol']
for col in outlier_cols:
    cap_value = df[col].quantile(0.99)
    df[col] = np.where(df[col] > cap_value, cap_value, df[col])

# 5. Categorical Encoding
df['koi_disposition'] = df['koi_disposition'].astype('category').cat.codes
df['koi_pdisposition'] = df['koi_pdisposition'].astype('category').cat.codes
df = pd.get_dummies(df, columns=['koi_tce_delivname'], drop_first=True)

df = df.dropna()
print(f"Remaining missing values: {df.isnull().sum().sum()}")
print(f"Final dataset shape: {df.shape}")

# -----------------------------------------------------------------------------

# 6. Export to Cleaned CSV
df.to_csv(output_path, index=False)

print(f"Data cleaning complete. Cleaned file saved securely at: {output_path}")

Remaining missing values: 0
Final dataset shape: (8945, 46)
Data cleaning complete. Cleaned file saved securely at: 01_NASA_Exoplanet_Archive_(Astrophysics)/cleaned_cumulative.csv


In [6]:
# BLOCK-05
import subprocess

# ==========================================
# Configuration: Update these variables
# ==========================================

FOLDER_PATH = "/content/drive/MyDrive/data-cleaning"   # The folder you want to push (e.g., /content/drive/MyDrive/my_project)
GITHUB_USERNAME = "arinskyyyy"                         # Your GitHub username
REPO_NAME = "data-cleaning"                            # The destination repository name
COMMIT_MESSAGE = "Update files via Colab"              # Commit message
USER_EMAIL = "arinskyyyy@gmail.com"                    # The email tied to your GitHub account
USER_NAME = "arinskyyyy"                               # Your display name
BRANCH = "main"                                        # The branch you are pushing to (usually main or master)

# ==========================================

def run_command(command):
    """Helper function to run shell commands and print the output."""
    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    if result.returncode == 0:
        print(f"✅ {result.stdout.strip()}")
    else:
        print(f"❌ Error: {result.stderr.strip()}")

# 1. Retrieve the token securely from Colab Secrets
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception as e:
    raise RuntimeError("Could not find GITHUB_TOKEN. Please add it to Colab Secrets (the key icon on the left).")

# Construct the secure remote URL
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

# 2. Navigate to the target folder
if not os.path.exists(FOLDER_PATH):
    raise FileNotFoundError(f"The folder {FOLDER_PATH} does not exist.")
os.chdir(FOLDER_PATH)
print(f"Changed directory to: {os.getcwd()}")

# 3. Execute Git Commands
print("\nConfiguring Git...")
run_command("git init")
run_command(f'git config --global user.email "{USER_EMAIL}"')
run_command(f'git config --global user.name "{USER_NAME}"')

print("\nStaging and Committing...")
run_command("git add .")
# Note: If there are no changes, commit will throw an error, which is normal.
run_command(f'git commit -m "{COMMIT_MESSAGE}"')

print("\nPushing to GitHub...")
# Remove existing origin if it exists to avoid conflicts, then add the secure one
run_command("git remote remove origin")
run_command(f"git remote add origin {REPO_URL}")
run_command(f"git branch -M {BRANCH}")

# Push changes
run_command(f"git push -u origin {BRANCH}")

NameError: name 'os' is not defined

In [ ]:
%%writefile .gitignore
# Ignore Python cache
__pycache__/
*.pyc

# Ignore dataset files

# Ignore specific folders
my_private_data/

Overwriting .gitignore
